# AcneClear AI - YOLO11n acne lesion training

Dataset: https://universe.roboflow.com/hadids/multi-class-acne

Run this notebook in Google Colab with a GPU runtime. It downloads the Roboflow object-detection export, verifies class IDs, splits, image-label pairing and bounding boxes, trains YOLO11n, evaluates on the test split and exports the real `best.pt`. No model is generated by the backend.

In [ ]:
%pip install -q ultralytics roboflow pyyaml


In [ ]:
import torch
from pathlib import Path

assert torch.cuda.is_available(), "Set the Colab runtime to GPU before running this notebook"
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = ""
DATASET_VERSION = 1

assert ROBOFLOW_API_KEY, "Set ROBOFLOW_API_KEY in this cell"
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("hadids").project("multi-class-acne")
version = project.version(DATASET_VERSION)
dataset = version.download("yolov8")
print("dataset:", dataset.location)


## Verify dataset configuration

The export must be object detection in YOLO format with exactly six classes at IDs 0-5: Blackheads, Whiteheads, Papules, Pustules, Nodules, Cysts.

In [ ]:
import yaml
from pathlib import Path

root = Path(dataset.location)
cfg_path = root / "data.yaml"
cfg = yaml.safe_load(cfg_path.read_text())
raw_names = cfg["names"]
if isinstance(raw_names, dict):
    names = [str(raw_names.get(i, raw_names.get(str(i)))) for i in range(len(raw_names))]
else:
    names = [str(item) for item in raw_names]
expected = ["Blackheads", "Whiteheads", "Papules", "Pustules", "Nodules", "Cysts"]
assert names == expected, f"Class mismatch: {names} != {expected}"
if "val" not in cfg and "valid" in cfg:
    cfg["val"] = cfg.pop("valid")
cfg["names"] = names
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("data.yaml:", cfg)


In [ ]:
import shutil
from collections import Counter
from pathlib import Path

def split_paths(split):
    first = (root / split / "images", root / split / "labels")
    second = (root / "images" / split, root / "labels" / split)
    return first if first[0].exists() else second

for split in ("train", "val", "test"):
    images_dir, labels_dir = split_paths(split)
    print(split, images_dir, labels_dir)
    assert images_dir.exists() and labels_dir.exists(), f"Missing split {split}"

test_images, test_labels = split_paths("test")
if not list(test_images.glob("*")):
    val_images, val_labels = split_paths("val")
    test_images.mkdir(parents=True, exist_ok=True)
    test_labels.mkdir(parents=True, exist_ok=True)
    moved = 0
    for image_path in sorted(val_images.iterdir()):
        if not image_path.is_file():
            continue
        if moved % 2 == 1:
            label_path = val_labels / (image_path.stem + ".txt")
            shutil.move(str(image_path), str(test_images / image_path.name))
            if label_path.exists():
                shutil.move(str(label_path), str(test_labels / label_path.name))
        moved += 1
    cfg["test"] = str(test_images)
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print("Created test split from validation images")

bad = []
counts = Counter()
image_counts = {}
for split in ("train", "val", "test"):
    images_dir, labels_dir = split_paths(split)
    images = [item for item in images_dir.iterdir() if item.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    labels = list(labels_dir.glob("*.txt"))
    image_counts[split] = len(images)
    assert images, f"No images in {split}"
    missing = {item.stem for item in images} - {item.stem for item in labels}
    extra = {item.stem for item in labels} - {item.stem for item in images}
    if missing or extra:
        bad.append((split, "pairing", len(missing), len(extra)))
    for label_file in labels:
        for line_no, line in enumerate(label_file.read_text().splitlines(), 1):
            parts = line.split()
            if len(parts) != 5:
                bad.append((label_file.name, line_no, "token_count"))
                continue
            try:
                cls = int(parts[0])
                cx, cy, width, height = (float(value) for value in parts[1:])
            except ValueError:
                bad.append((label_file.name, line_no, "numeric"))
                continue
            if not 0 <= cls < len(names):
                bad.append((label_file.name, line_no, "class_id"))
                continue
            if not all(0.0 <= value <= 1.0 for value in (cx, cy, width, height)):
                bad.append((label_file.name, line_no, "range"))
            if width <= 0 or height <= 0 or width > 1 or height > 1:
                bad.append((label_file.name, line_no, "box_size"))
            if cx - width / 2 < -1e-6 or cx + width / 2 > 1 + 1e-6:
                bad.append((label_file.name, line_no, "x_bounds"))
            if cy - height / 2 < -1e-6 or cy + height / 2 > 1 + 1e-6:
                bad.append((label_file.name, line_no, "y_bounds"))
            counts[names[cls]] += 1
print("images per split:", image_counts)
print("annotations per class:", dict(counts))
assert all(counts[name] > 0 for name in names), "At least one class has no annotations"
assert not bad, f"Invalid annotations: {bad[:20]}"


## Train YOLO11n

Training uses the verified YOLO export, a fixed seed, early stopping and documented augmentation. Validation runs during training; final reported metrics come from the separate test split below.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
results = model.train(
    data=str(cfg_path),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    name="acneclear_yolo11n",
    patience=20,
    seed=42,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    close_mosaic=10,
    amp=True,
    val=True,
)
print("training completed:", results)


In [ ]:
import json
from pathlib import Path

metrics = model.val(data=str(cfg_path), split="test", device=0, imgsz=640)
map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)
assert 0.0 <= map50 <= 1.0 and 0.0 <= map50_95 <= 1.0
per_class_ap50 = [float(value) for value in getattr(metrics.box, "ap50", [])]
summary = {
    "dataset": "https://universe.roboflow.com/hadids/multi-class-acne",
    "dataset_version": DATASET_VERSION,
    "model": "yolo11n",
    "split": "test",
    "mAP50": map50,
    "mAP50-95": map50_95,
    "per_class_ap50": dict(zip(names, per_class_ap50)),
}
metrics_path = Path("/content/acneclear_training_metrics.json")
metrics_path.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))


In [ ]:
import shutil
from pathlib import Path

best = Path(model.trainer.best)
target = Path("/content/best.pt")
shutil.copy2(best, target)
print("checkpoint:", target)
print("metrics:", "/content/acneclear_training_metrics.json")
try:
    from google.colab import files
    files.download(str(target))
    files.download("/content/acneclear_training_metrics.json")
except ImportError:
    print("Not running in Colab: copy the printed files manually.")


## Remaining external action

Copy the downloaded `best.pt` to `backend/models/best.pt` in this project, then restart the backend and confirm `GET /api/detect/status` reports `yolo_active: true`. Report only the `mAP50` and `mAP50-95` values printed by the evaluation cell. Do not copy a checkpoint whose class order differs from the six required classes.